# Volume 5, Chapter 2: Arithmetic functions on average (Python companion)

Exercises first, solutions at the end. We sieve $\tau$, $\sigma$, $\varphi$, $\mu$ and $\omega$ up to $10^6$ at once, compare averages with medians, check Wintner's mean values and the error terms of the chapter, count coprime tuples and lattice points in discs, find the constant terms that the chapter leaves as $O(x)$, and test Cramér's mean-square law for the divisor problem. Requirements: numpy, mpmath, matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P2.1 (★) Average versus typical.** Compute $\tau(n)$ for all $n\le10^6$ with a divisor sieve. Compare the mean with Volume 1's $\log x+2\gamma-1$, and find the median, the quartiles and the proportion of $n$ with at least $14$ divisors (the think box). Compare with $(\log x)^{\log2}$.

**P2.2 (★) Mean values.** Check Corollary 2.4 and Exercise 2.2: the means of $\mu^2$, $\varphi(n)/n$, $\sigma(n)/n$ and $n/\varphi(n)$ up to $10^6$, and the Euler product $\prod_p(1+\frac1{p(p-1)})$.

**P2.3 (★★) Error terms.** For $x=10^3,\dots,10^6$, tabulate $\Delta(x)$, the errors of Theorem 2.6 divided by their claimed sizes, and the proportion of coprime pairs (Corollary 2.7). Then check Exercise 2.3 for $k=2,3,4$ and Exercise 2.4 for $r$ up to $10^4$.

**P2.4 (★★) The next constants.** Theorem 2.10 and Exercise 2.5 stop at $O(x)$. Compute the residue of $\zeta(s)^3\frac{x^s}s$ at $s=1$ using $\zeta(s)=\frac1{s-1}+\gamma-\gamma_1(s-1)+\cdots$ to predict the next term $cx$ for $\tau_3$, and check $c=\frac6{\pi^2}\bigl(2\gamma-1-2\frac{\zeta'(2)}{\zeta(2)}\bigr)$ for $2^{\omega}$.

**P2.5 (★★) Cramér's law.** Compute $\int_1^X\Delta(x)^2\,\mathrm dx$ for $X=10^4,10^5,10^6$ (on each $[k,k+1[$, $\Delta$ is explicit) and compare with $cX^{3/2}$, $c=\frac{\zeta(3/2)^4}{6\pi^2\zeta(3)}$, as in the remark on the divisor problem. Plot $\Delta(x)/x^{1/4}$ against $\sqrt x$ for $10^5\le x\le10^6$, as in Figure 2.2.

## Solutions

### P2.1

In [ ]:
import math
import numpy as np
import mpmath as mp

N = 10**6
n = np.arange(N + 1)
tau = np.zeros(N + 1, dtype=np.int64)
sigma = np.zeros(N + 1, dtype=np.int64)
for d in range(1, N + 1):
    tau[d::d] += 1
    sigma[d::d] += d
phi = n.astype(np.float64)
mu = np.ones(N + 1, dtype=np.int64)
omega = np.zeros(N + 1, dtype=np.int64)
is_p = np.ones(N + 1, dtype=bool); is_p[:2] = False
for p in range(2, N + 1):
    if is_p[p]:
        is_p[2 * p::p] = False
        phi[p::p] *= 1 - 1 / p
        mu[p::p] *= -1
        mu[p * p::p * p] = 0
        omega[p::p] += 1
phi = np.rint(phi).astype(np.int64)
mu[0] = 0
print("tau(720720) =", tau[720720], "  phi(1000) =", phi[1000], "  sigma(28) =", sigma[28], "  mu(30) =", mu[30])

a = tau[1:]
print(f"tau(n), n <= 1e6: mean {a.mean():.4f} (log x + 2 gamma - 1 = {math.log(N) + 2 * float(mp.euler) - 1:.4f}),"
      f" median {np.median(a):.0f}, share with tau >= 14: {(a >= 14).mean():.3f}")
print("quartiles and 90th, 99th percentiles:", np.percentile(a, [25, 75, 90, 99]), "  (log 1e6)^(log 2) =", round(math.log(N) ** math.log(2), 2))

The mean is $13.970$, matching $\log x+2\gamma-1$ to four digits, but the median is $8$, a quarter of the integers have at most $4$ divisors, and only $34\%$ reach the mean. The top $1\%$ have $72$ divisors or more. The typical size $(\log x)^{\log2}\approx6.2$ is close to the median: the mean is carried by rare integers with many small prime factors.

### P2.2

In [ ]:
m = n[1:].astype(float)
print("mean of mu^2      :", (mu[1:] != 0).mean(), "  6/pi^2 =", 6 / math.pi**2)
print("mean of phi(n)/n  :", (phi[1:] / m).mean())
print("mean of sigma(n)/n:", (sigma[1:] / m).mean(), "  pi^2/6 =", math.pi**2 / 6)
print("mean of n/phi(n)  :", (m / phi[1:]).mean(), "  zeta(2)zeta(3)/zeta(6) =", float(mp.zeta(2) * mp.zeta(3) / mp.zeta(6)))
# the Euler product, as a "probability model": prod over p <= P of E[factor at p]
P = [p for p in range(2, 10**5) if is_p[p]]
print("prod_{p < 1e5} (1 + 1/(p(p-1))) =", math.prod(1 + 1 / (p * (p - 1)) for p in P))

All four means agree with their predictions to about five digits at $10^6$. For $\varphi(n)/n$ the error is about $10^{-9}$, much better than the $O(\frac{\log x}x)$ of Exercise 2.1. The Euler product over $p<10^5$ is already within $2\cdot10^{-6}$ of $\frac{\zeta(2)\zeta(3)}{\zeta(6)}$, since its tail is about $\sum_{p>P}\frac1{p^2}\approx\frac1{P\log P}$.

### P2.3

In [ ]:
g = float(mp.euler)
print("    x      Delta(x)    sigma: err/(x log x)   phi: err/(x log x)   Q: err/sqrt(x)   coprime pairs / x^2")
cum_tau, cum_sig, cum_phi = np.cumsum(tau), np.cumsum(sigma), np.cumsum(phi)
cum_Q = np.cumsum(mu != 0)
for e in range(3, 7):
    x = 10**e
    L = math.log(x)
    print(f"  1e{e}   {cum_tau[x] - (x * L + (2 * g - 1) * x):9.2f}      {(cum_sig[x] - math.pi**2 / 12 * x * x) / (x * L):+.4f}"
          f"               {(cum_phi[x] - 3 / math.pi**2 * x * x) / (x * L):+.4f}            {(cum_Q[x] - 6 / math.pi**2 * x) / math.sqrt(x):+.4f}"
          f"         {(2 * cum_phi[x] - 1) / x**2:.6f}")

In [ ]:
for k in (2, 3, 4):
    x = 1000
    C = sum(int(mu[d]) * (x // d) ** k for d in range(1, x + 1))
    print(f"k = {k}: C_k(1000)/1000^k = {C / x**k:.6f}   1/zeta(k) = {1 / float(mp.zeta(k)):.6f}")
for r in (10, 100, 1000, 10000):
    cnt = sum(2 * math.isqrt(r * r - b * b) + 1 for b in range(-r, r + 1))
    print(f"circle r = {r:5d}: N(r) - pi r^2 = {cnt - math.pi * r * r:9.2f}   bound pi(sqrt2 r + 1/2) = {math.pi * (math.sqrt(2) * r + 0.5):9.1f}")

$\Delta(x)$ stays below $100$ while $\sqrt x$ reaches $1000$. The $\sigma$ error is genuinely of size $x\log x$ (the ratio hovers near $0.08$), while the $\varphi$ error is much smaller than $x\log x$ for these $x$ and the squarefree error is a tiny fraction of $\sqrt x$. The coprime tuples converge to $\frac1{\zeta(k)}$ at the predicted speeds, and the circle errors ($-212$ at $r=10^4$) are about $200$ times smaller than the bound of Exercise 2.4, consistent with the conjectured $O(r^{1/2+\varepsilon})$.

### P2.4

In [ ]:
tau3 = np.zeros(N + 1, dtype=np.int64)
for d in range(1, N + 1):
    tau3[d::d] += tau[d]
c3 = np.cumsum(tau3)
g1 = float(mp.stieltjes(1))
print("gamma_1 =", g1, "  predicted constant term of P_3:", 1 - 3 * g + 3 * g * g - 3 * g1)
for e in range(3, 7):
    x = 10**e
    L = math.log(x)
    print(f"x = 1e{e}: (sum tau_3 - x(L^2/2 + (3 gamma - 1) L)) / x = {(c3[x] - x * (L * L / 2 + (3 * g - 1) * L)) / x:.4f}")
W = np.cumsum(2.0 ** omega)
C2 = 6 / math.pi**2 * (2 * g - 1 - 2 * float(mp.zeta(2, derivative=1) / mp.zeta(2)))
for e in range(3, 7):
    x = 10**e
    print(f"x = 1e{e}: (sum 2^omega - (6/pi^2) x log x) / x = {(W[x] - 6 / math.pi**2 * x * math.log(x)) / x:.5f}   predicted {C2:.5f}")

Expanding $\zeta(s)^3=u^{-3}+3\gamma u^{-2}+(3\gamma^2-3\gamma_1)u^{-1}+\cdots$ and $\frac{x^s}s=x\bigl(1+(L-1)u+(\frac{L^2}2-L+1)u^2+\cdots\bigr)$, with $u=s-1$ and $L=\log x$, the residue is $x\bigl(\frac{L^2}2+(3\gamma-1)L+1-3\gamma+3\gamma^2-3\gamma_1\bigr)$: the main term of Theorem 2.10 plus $0.4863x$. The data approach it slowly ($0.488$ at $10^6$), as the error term $O(x^{2/3}\log x)$ allows. For $2^\omega$ the prediction $0.78687$ is matched to five digits at $10^6$; it comes from the hyperbola method with the expansion $\sum_{d\le y}\frac{\mu^2(d)}d=\frac6{\pi^2}\bigl(\log y+\gamma-2\frac{\zeta'(2)}{\zeta(2)}\bigr)+O(y^{-1/2}\log y)$.

### P2.5

In [ ]:
import matplotlib.pyplot as plt
X = 10**6
x_int = np.arange(1, X + 1, dtype=float)
Delta_at = cum_tau[1:X + 1] - (x_int * np.log(x_int) + (2 * g - 1) * x_int)
# mean square of Delta, against Cramer: int_1^X Delta^2 ~ c X^(3/2)
c_cramer = float(mp.zeta(1.5)**4 / mp.zeta(3) / (6 * mp.pi**2))
# Delta on [k, k+1) is Delta(k) - (f(t) - f(k)) with f(t) = t log t + (2 gamma - 1) t; integrate numerically with 8 points per unit
t = np.linspace(0, 1, 9)[:-1] + 1 / 16
ints = []
for X2 in (10**4, 10**5, 10**6):
    k = np.arange(1, X2, dtype=float)
    vals = Delta_at[:X2 - 1, None] - ((k[:, None] + t) * np.log(k[:, None] + t) + (2 * g - 1) * (k[:, None] + t) - (k[:, None] * np.log(k[:, None]) + (2 * g - 1) * k[:, None]))
    I = (vals**2).mean(axis=1).sum()
    print(f"X = {X2:8d}: int_1^X Delta^2 / X^1.5 = {I / X2**1.5:.4f}   Cramer's constant {c_cramer:.4f}")
fig, ax = plt.subplots(figsize=(9, 3))
sel = np.arange(10**5, X, 37)
ax.plot(np.sqrt(sel), Delta_at[sel - 1] / sel**0.25, lw=0.3)
ax.set_xlabel("sqrt(x)"); ax.set_ylabel("Delta(x) / x^(1/4)")
plt.show()

The normalised mean square climbs towards Cramér's constant: $0.602$, $0.633$, $0.645$ against $0.654$, with a slowly decaying correction. So $\Delta(x)$ has root mean square about $0.99\,x^{1/4}$ (the derivative of $cX^{3/2}$ is $\frac32cX^{1/2}$, and $\sqrt{1.5\times0.654}=0.99$). The plot looks like stationary noise of amplitude $1$ to $3$: in the variable $\sqrt x$, Voronoï's formula writes $\Delta(x)/x^{1/4}$ as a sum of waves $\cos(4\pi\sqrt{nx}-\frac\pi4)$ with amplitudes $\frac{\tau(n)}{\pi\sqrt2\,n^{3/4}}$, a kind of music of the divisors.